# Zero2FSD — Week 1 — Driving ML Gym

This notebook is the **lecture path** for Week 1.

- **Scaffold** (~60–70%): data loading, model, cross-entropy training loop
- **Fill**: implement `focal_loss` and `minority_recall` in the module `.py` files
- **From scratch**: implement `build_error_gallery` yourself

Optional autograd appendix: `modules/00_nn_scratch` (treat as **appendix 00b**, not the default chapter).

The crops are **synthetic checked-in PNGs** under `data/m00_sample` (CC0 license), not random `torch.randn` tensors.

## Session card

Zero2FSD · Week 1

**Today's win:** See pedestrian recall, not just accuracy, on the checked-in crops.

**Time:** 25 / 55 / 90 minutes

A day counts when you export an artifact or pass the tests for a fill you wrote. Opening the notebook does not. Set pause_week to true if you need a week off; the count stays where it is.

**Your stack so far**

- [ ] m00 — Driving ML Gym
- [ ] m01 — Cameras & IPM
- [ ] m02 — HydraNet
- [ ] m03 — BEV transform
- [ ] m04 — Occupancy
- [ ] m05 — Vector tracking
- [ ] m06 — Planning
- [ ] m07 — Control
- [ ] m08 — Capstone
- [ ] m09 — System architecture

XP is not awarded for opening this notebook.

In [ ]:
import sys
from pathlib import Path

repo = Path.cwd()
if not (repo / 'modules' / '00_ml_gym').exists():
    repo = repo.parent
sys.path.insert(0, str(repo / 'modules'))
from common.progress import format_stack

progress_path = repo / 'artifacts' / 'progress.json'
if progress_path.is_file():
    import json
    with progress_path.open() as f:
        progress = json.load(f)
    print(format_stack(progress))
else:
    print('No progress file yet. Opening this notebook awards 0 XP.')

**Principle 1 — Model as function.**

A classifier is a function from a crop to one score per class. Those scores are not probabilities until you apply softmax. Learning changes the weights so a chosen loss gets smaller on the training distribution. If that distribution is mostly empty road, the function gets good at road unless the loss says otherwise. This module is a function, a loss, and a dataset — not a claim that the net "understands driving."

**Principle 2 — Loss defines good.**

The loss is the definition of "good" that gradient descent sees. Cross-entropy charges `-log(p_true)` on every crop, so a pile of easy road crops can outweigh a few pedestrians. Focal loss multiplies that charge by `(1 - p_true)^gamma`. When the model is already sure and correct, the factor goes to 0 and the easy crop stops dominating. When gamma is 0 the factor is 1, so focal loss is cross-entropy if there is no extra class weight. You will check that equality in code, not by trusting the function name.

**Principle 3 — Inspect errors.**

One accuracy number averages over classes. With many road crops and few pedestrians, a model that never predicts pedestrian can still look mostly right. A gallery of mistakes, sorted by how confident the wrong answer was, shows whether the model is confused or confidently wrong. Confident mistakes are the ones you would queue for more labels. You will build that gallery yourself; the scaffold does not do it for you.

In [ ]:
import sys
from pathlib import Path
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

repo = Path.cwd()
if not (repo / 'modules' / '00_ml_gym').exists():
    repo = repo.parent
sys.path.insert(0, str(repo / 'modules' / '00_ml_gym'))
torch.manual_seed(0)

from config import TrainConfig
from dataset import CLASS_NAMES, get_dataloaders
from model import DrivingClassifier
from losses import cross_entropy_loss, focal_loss
from metrics import accuracy, per_class_recall, minority_recall
print('repo:', repo)

**Tensor shapes.**

**B** is the batch size — how many crops are processed together. **C** is the number of color channels (3 for RGB). **H** is image height in pixels. **W** is image width in pixels. PyTorch expects image batches as `(B, C, H, W)` while PNG files on disk are stored as `(H, W, C)`.

In [ ]:
cfg = TrainConfig(data_dir=repo / 'data' / 'm00_sample')
train_loader, val_loader = get_dataloaders(cfg.data_dir, batch_size=16, seed=0)
images, labels = next(iter(train_loader))
assert images.shape[1:] == (3, 64, 64), images.shape
assert labels.shape == (images.shape[0],), labels.shape
print('batch', images.shape, 'labels', labels.shape)
print('dtype', images.dtype, labels.dtype)
print('pixel min/max', images.min().item(), images.max().item(), '(real PNG pixels in [0,1], not randn)')

In [ ]:
from PIL import Image
import numpy as np
ds = train_loader.dataset
fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for ax, i in zip(axes.flat, range(8)):
    img, lab = ds[i]
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(CLASS_NAMES[lab.item()], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample crops (from PNG files)')
plt.tight_layout()
plt.show()

In [ ]:
counts = train_loader.dataset.class_counts()
n = sum(counts.values())
count_road = counts[0]
always_road_acc = count_road / n
plt.bar(CLASS_NAMES, [counts[i] for i in range(4)])
plt.title('Train class counts — accuracy will lie')
plt.xticks(rotation=20)
plt.show()
print(f'Always-predict-clear_road accuracy: {always_road_acc:.3f} ({count_road}/{n})')
print('Pedestrian recall of that classifier: 0.0 — the accuracy lie.')

**Derive cross-entropy.** Softmax turns logits `z_k` into class probabilities:

$$p_k = \frac{\exp(z_k)}{\sum_j \exp(z_j)}$$

Cross-entropy for true class $y$ is $-\log p_y$. Minimizing CE is the same as maximizing the probability assigned to the true class.

In [ ]:
import torch.nn.functional as F
z = torch.tensor([2.0, 1.0, 0.5, -1.0])
y = torch.tensor(0)
manual = -torch.log(F.softmax(z, dim=0)[y])
ce = F.cross_entropy(z.unsqueeze(0), y.unsqueeze(0))
print('manual CE', manual.item(), 'F.cross_entropy', ce.item())
assert torch.allclose(manual, ce, atol=1e-6)

In [ ]:
model = DrivingClassifier(num_classes=4)
logits = model(images)
assert logits.shape == (images.shape[0], 4)
n_params = sum(p.numel() for p in model.parameters())
print('logits', logits.shape, 'parameter count', n_params)

You are about to fit **cross-entropy** for four epochs. Watch **per-class recall**, not only accuracy. Pedestrian is class index **2** (`CLASS_NAMES[2]`).

In [ ]:
from train import train_epoch, evaluate
import torch.optim as optim
import numpy as np

device = torch.device('cpu')
model = DrivingClassifier().to(device)
opt = optim.AdamW(model.parameters(), lr=1e-3)
for ep in range(4):
    train_epoch(model, train_loader, opt, cross_entropy_loss, device)
    loss, acc, recalls = evaluate(model, val_loader, cross_entropy_loss, device, 4)
    print(f'epoch {ep+1} val_acc={acc:.3f} recalls={[round(r, 2) for r in recalls]}')
ce_ped_recall = recalls[2]

# Confusion matrix on val split
num_classes = 4
cm = np.zeros((num_classes, num_classes), dtype=np.int64)
model.eval()
with torch.no_grad():
    for imgs, tgts in val_loader:
        preds = model(imgs.to(device)).argmax(dim=-1).cpu().numpy()
        for p, t in zip(preds, tgts.numpy()):
            cm[t, p] += 1
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(num_classes), CLASS_NAMES, rotation=45, ha='right')
ax.set_yticks(range(num_classes), CLASS_NAMES)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
fig.colorbar(im, ax=ax, fraction=0.046)
plt.title('Validation confusion matrix (CE)')
plt.tight_layout()
plt.show()

**Focal loss derivation.** Focal loss modulates cross-entropy with a focusing factor:

$$\mathrm{FL} = -(1 - p_t)^\gamma \log(p_t)$$

When $\gamma = 0$ the factor is 1, so focal loss reduces to cross-entropy. Numeric sketch: if $p_t = 0.99$ and $\gamma = 2$, the factor is $0.0001$ — an easy correct crop barely contributes. If $p_t = 0.3$ and $\gamma = 2$, the factor is about $0.49$ — a hard example still matters. $\alpha$ is an optional per-class weight tensor of shape `(K,)` applied as `alpha[targets]`.

**FILL — `focal_loss`** in `modules/00_ml_gym/losses.py` only.

Inputs: logits `(B, K)` and targets `(B,)`. Do **not** paste a solution into this notebook. Hint: the stable route is `log_softmax`, but implement the function body in the `.py` file.

In [ ]:
try:
    torch.manual_seed(0)
    z = torch.randn(4, 4)
    t = torch.tensor([0, 1, 2, 3])
    fl0 = focal_loss(z, t, gamma=0.0)
    ce = cross_entropy_loss(z, t)
    print('gamma=0 allclose', torch.allclose(fl0, ce, atol=1e-5))
    assert torch.allclose(fl0, ce, atol=1e-5)
    z2 = torch.tensor([[8.0, 0.0, 0.0, 0.0]])
    t2 = torch.tensor([0])
    fl2 = focal_loss(z2, t2, gamma=2.0).item()
    ce2 = cross_entropy_loss(z2, t2).item()
    print('gamma=2 confident FL', fl2, 'CE', ce2, 'FL < CE', fl2 < ce2)
    assert fl2 < ce2
except NotImplementedError:
    print('STOP: implement focal_loss in modules/00_ml_gym/losses.py')

In [ ]:
try:
    focal_loss(torch.randn(2, 4), torch.tensor([0, 1]), gamma=2.0)
    torch.manual_seed(0)
    model_fl = DrivingClassifier().to(device)
    opt_fl = optim.AdamW(model_fl.parameters(), lr=1e-3)
    focal_fn = lambda lg, tg: focal_loss(lg, tg, gamma=2.0)
    for ep in range(4):
        train_epoch(model_fl, train_loader, opt_fl, focal_fn, device)
        _, acc_f, recalls_f = evaluate(model_fl, val_loader, focal_fn, device, 4)
        print(f'focal epoch {ep+1} val_acc={acc_f:.3f} recalls={[round(r, 2) for r in recalls_f]}')
    print(f'CE pedestrian recall (index 2): {ce_ped_recall:.3f}')
    print(f'Focal pedestrian recall (index 2): {recalls_f[2]:.3f}')
    print(f'Delta (focal - CE): {recalls_f[2] - ce_ped_recall:.3f}')
except NotImplementedError:
    print('Comparison waits until focal_loss fill is done')

**FILL — `minority_recall`** in `modules/00_ml_gym/metrics.py`.

Recall = TP / support for one class; if support is 0, return 0. Hand example: preds `[2, 0, 2, 1]`, targets `[2, 2, 0, 2]`, minority class 2 → recall **1/3**.

In [ ]:
try:
    preds = torch.tensor([2, 0, 2, 1])
    tgts = torch.tensor([2, 2, 0, 2])
    mr = minority_recall(preds, tgts, minority_class=2)
    print('minority_recall', mr)
    assert abs(mr - 1/3) < 1e-6
except NotImplementedError:
    print('STOP: implement minority_recall in metrics.py')

**FROM SCRATCH — `build_error_gallery`** in `modules/00_ml_gym/error_gallery.py`.

Contract: images NCHW in `[0, 1]`, targets `(N,)`, logits `(N, K)`. Sort mistakes by descending confidence of the **wrong** class. Return a dict with keys `path`, `n_errors`, `order`; write a figure even when `n_errors=0`. Also add a test or un-skip behavior in `tests/test_student_fills.py` by implementing the function. Do not implement it in this notebook.

In [ ]:
from error_gallery import build_error_gallery
try:
    out = repo / 'artifacts' / 'm00' / 'nb_gallery.png'
    model.eval()
    imgs, tgts = next(iter(val_loader))
    with torch.no_grad():
        lg = model(imgs)
    res = build_error_gallery(imgs, tgts, lg, CLASS_NAMES, out)
    print(res)
    plt.imshow(np.array(Image.open(out)))
    plt.axis('off')
    plt.title('Error gallery')
    plt.show()
except NotImplementedError:
    print('STOP: implement build_error_gallery in error_gallery.py')

**Free response (Principle 2):** Which principle did focal loss apply? Why does $\gamma=0$ match cross-entropy?

_Write 3–6 sentences here before you open `solutions/00_ml_gym`._

**Free response (Principle 3):** The always-road classifier had high accuracy and zero pedestrian recall. Which principle says that metric was the wrong definition of good?

_Write 3–6 sentences here._

In [ ]:
from break_it_fix_it import main as break_demo
break_demo()

**Tests**

```bash
python3 -m pytest modules/00_ml_gym -q
```

Scaffold tests in `test_scaffold.py` always run and verify the provided code. `test_assignment_solutions.py` imports reference code from `solutions/00_ml_gym`; student fill tests in `test_student_fills.py` skip until you implement the function and fail if the implementation is wrong.

In [ ]:
from train import main as train_main
metrics = train_main(TrainConfig(
    data_dir=repo / 'data' / 'm00_sample',
    epochs=2,
    loss_name='cross_entropy',
))
metrics_path = repo / 'artifacts' / 'm00' / metrics['run_id'] / 'metrics.json'
print('metrics path:', metrics_path)
print('minority_recall field:', metrics['minority_recall'])

**Come back cue**

Tomorrow: 25-min error-gallery review — implement build_error_gallery and re-run that cell.

Suggested slot: 25 minutes. 55 or 90 if you are also writing the principle cells.

In [ ]:
import sys
import json
from pathlib import Path

repo = Path.cwd()
if not (repo / 'modules' / '00_ml_gym').exists():
    repo = repo.parent
sys.path.insert(0, str(repo / 'modules'))
from common.progress import come_back_cue

print(come_back_cue('m00'))
progress_path = repo / 'artifacts' / 'progress.json'
if progress_path.is_file():
    with progress_path.open() as f:
        xp = json.load(f).get('xp', 0)
    print(f'XP so far: {xp}')